> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 5 · Notebook 02 — Streaming indicators

**Sessions:** S3 (Streaming, incremental indicators) · [Lesson plan](../../docs/lessons/PART_05_ANALYTICS_LIBRARY.md) · graded labs in [`labs/part05/`](../../labs/part05/)

**You will:**
1. Turn a vectorized SMA and EMA into O(1) streaming updates.
2. Keep a rolling maximum with a monotonic deque.
3. Prove streaming equals vectorized on every prefix, with a property test.
4. See the cost of recomputing, and the bug of updating on every tick.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with a known structure, so you always know which effects are real and which are luck.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p5lib.py is in notebooks/part05/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p5lib as p

p.use_course_style()
from collections import deque
import time

## 1. Why streaming

Live, a strategy gets one new bar at a time. Recomputing an indicator over the whole history on every bar costs O(n) per bar, O(n²) per day. A **streaming** indicator keeps a little state and updates in O(1). The contract: its output equals the vectorized version on **every prefix** of the data.

In [ ]:
df = p.synthetic_ohlcv(3000, seed=1)
o, h, l, c, v = p.arrays(df)

t0 = time.perf_counter()
recomputed = [p.ema(c[: i + 1], 20)[-1] for i in range(len(c))]
t_re = time.perf_counter() - t0
t0 = time.perf_counter()
streamed = p.stream(p.StreamingEMA(20), c)
t_st = time.perf_counter() - t0
print(f"recompute every bar: {t_re:.2f} s   streaming: {t_st * 1000:.1f} ms   ({t_re / t_st:,.0f}× faster)")
print("same values:", np.allclose(recomputed, streamed, equal_nan=True))

## 2. A streaming SMA

Keep the last `n` values in a `deque` and a running total: add the new value, and once there are more than `n`, remove the oldest from both. Return `None` until the window is full.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
class MyStreamingSMA:
    def __init__(self, n):
        self.n, self.buf, self.total = n, deque(), 0.0

    def update(self, x):
        self.buf.append(x)
        self.total += x
        if len(self.buf) > self.n:
            self.total -= self.buf.popleft()
        return self.total / self.n if len(self.buf) == self.n else None

mine = p.stream(MyStreamingSMA(20), c)
mine = p.check("streaming SMA", mine, p.sma(c, 20))
mine[17:23].round(4)

## 3. A streaming EMA

Collect the first `n` values to compute the SMA seed; after that, apply the recursion to the stored value. Return `None` until the seed exists (so the output matches `p.ema`, NaN for the first `n − 1` bars).

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
class MyStreamingEMA:
    def __init__(self, n):
        self.n, self.alpha, self._seed, self.value = n, 2.0 / (n + 1), [], None

    def update(self, x):
        if self.value is None:
            self._seed.append(x)
            if len(self._seed) == self.n:
                self.value = sum(self._seed) / self.n
        else:
            self.value = self.alpha * x + (1 - self.alpha) * self.value
        return self.value

mine = p.attempt(p.stream, MyStreamingEMA(20), c)
mine = p.check("streaming EMA", mine, p.ema(c, 20))

## 4. Rolling maximum in O(1): the monotonic deque

Donchian channels and the Stochastic need the max of the last `n` highs. Rescanning the window is O(n) per bar. Instead keep a deque of `(index, value)` with **decreasing values**:
* before appending `x`, pop from the back every value `<= x` (they can never be the max while `x` is in the window);
* after appending, pop the front if its index has left the window (`index <= i − n`);
* the front is the max.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
class MyRollingMax:
    def __init__(self, n):
        self.n, self.i, self.q = n, -1, deque()

    def update(self, x):
        self.i += 1
        while self.q and self.q[-1][1] <= x:
            self.q.pop()
        self.q.append((self.i, x))
        if self.q[0][0] <= self.i - self.n:
            self.q.popleft()
        return self.q[0][1] if self.i >= self.n - 1 else None

mine = p.attempt(p.stream, MyRollingMax(20), h)
mine = p.check("rolling max", mine, pd.Series(h).rolling(20).max().to_numpy())

## 5. Prove it for every input: a property test

Examples check the data you picked. Hypothesis generates hundreds of arrays, including nasty ones (constant runs, huge jumps), and checks the streaming rolling standard deviation (Welford-style add/remove updates) against pandas on every prefix.

The first time we ran this with a fixed tolerance of 1e-6, Hypothesis found a failure within seconds, and shrank it to the simplest case: one huge value followed by a flat run.

In [ ]:
spike = np.array([3045.0] + [2.0] * 24)
got = p.stream(p.StreamingStd(20), spike)[19:23]
want = pd.Series(spike).rolling(20).std(ddof=0).to_numpy()[19:23]
print("streaming:", got)
print("pandas:   ", want)

When the spike leaves the window, the running sum of squares should drop back to exactly 0, but subtracting a number of about 9·10⁶ leaves rounding error behind, and the square root magnifies it near zero variance: 1e-5 on data of size 3·10³, a relative error of 3·10⁻⁹. That is floating point, not a logic bug. The test must say so explicitly: **the tolerance scales with the data**. (Production code also clamps a tiny negative variance to 0, or re-syncs from the buffer every `n` bars.)

In [ ]:
from hypothesis import given, settings, strategies as st
from hypothesis.extra.numpy import arrays

@settings(max_examples=200, deadline=None)
@given(arrays(np.float64, st.integers(25, 300), elements=st.floats(1, 1e4)))
def test_streaming_std_matches_pandas(x):
    got = p.stream(p.StreamingStd(20), x)
    want = pd.Series(x).rolling(20).std(ddof=0).to_numpy()
    np.testing.assert_allclose(got, want, rtol=1e-6, atol=1e-6 * np.abs(x).max(), equal_nan=True)

test_streaming_std_matches_pandas()
print("✔ 200 random arrays: streaming std == pandas rolling std on every prefix, within a scale-aware tolerance")

## 6. `update` on the close, `peek` in between

A live chart wants the indicator value of the **forming** bar on every tick. Calling `update` on each tick feeds the same bar into the state several times. `peek(price)` computes the value without changing the state; `update` runs once, on the bar close.

In [ ]:
rng = np.random.default_rng(0)
closes = c[:300]
wrong, right = p.StreamingEMA(20), p.StreamingEMA(20)
for px in closes:
    ticks = px * (1 + rng.normal(0, 0.002, 4))              # four intrabar ticks, then the close
    for t in ticks:
        wrong.update(t)                                     # the bug: every tick mutates the state
        right.peek(t)                                       # display only
    wrong.update(px)
    right.update(px)
ref = p.ema(closes, 20)[-1]
print(f"vectorized EMA of the closes: {ref:.4f}")
print(f"update on the close only:     {right.value:.4f}")
print(f"update on every tick:         {wrong.value:.4f}  ← a different indicator (an EMA of ticks), not the one you backtested")

## Wrap-up

* Streaming indicators: running sums, recursions, monotonic deques, Welford updates, all O(1) per bar.
* The equivalence test on every prefix is what lets you trust live values to match the backtest.
* `update` once per closed bar; `peek` for intrabar displays.
* Graded version: `labs/part05/week17_core` (`StreamingEMA`, `StreamingRSI`, `RollingMax`, `StreamingBollinger` with Hypothesis tests).